In [52]:
import pandas as pd
import requests
import yfinance as yf
import numpy as np
from fredapi import Fred

In [ ]:
headers = {"User-Agent": "Mozilla/5.0 (compatible; my-script/1.0)"}
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
html = requests.get(url, headers=headers).text
table = pd.read_html(html)[0]
table['Symbol'] = table['Symbol'].str.replace('.', '-').tolist()  # yfinance uses '-' not '.'
table

In [13]:
table.to_csv('../data/tickers.csv')

In [20]:
tickers = pd.read_csv('../data/tickers.csv')
tickers = tickers.drop(columns=['Unnamed: 0'])
tickers

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [26]:
# Download 10 years of daily adjusted close prices
prices = yf.download(
    tickers=tickers['Symbol'].to_list(),
    start='2016-01-01',
    end='2026-01-01',
    auto_adjust=True,       # use adjusted prices
    progress=False
)['Close']

prices.to_csv('../data/prices_raw.csv')
print(prices.shape)  # (trading_days, n_tickers)


(2514, 503)


In [32]:
prices.head()

Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2016-01-04,37.569214,23.730946,37.707539,NaN,35.380219,21.823122,86.944191,91.970001,44.534782,26.397755,...,20.020790,60.409386,26.027582,49.654362,31.806705,12.16,42.875286,91.126854,66.489998,43.515823
2016-01-05,37.439968,23.136261,37.550438,NaN,35.371990,21.902363,87.396690,92.339996,44.207561,26.692379,...,20.067774,61.551342,26.290037,50.077450,31.797894,11.51,42.768421,93.024414,64.820000,44.197056
2016-01-06,37.606152,22.683502,37.556999,NaN,35.075294,21.937229,87.225922,91.019997,42.326057,26.044203,...,19.624807,58.362644,26.567072,49.660782,31.401188,11.52,42.465591,93.427193,62.230000,44.206261
2016-01-07,36.008827,21.726154,37.445724,NaN,34.234665,21.915041,84.664482,89.110001,41.246212,25.425510,...,18.899946,52.872486,26.669157,48.865906,30.590141,11.16,41.016819,91.279053,59.410000,42.862213
2016-01-08,35.630291,21.841030,36.424656,NaN,33.517670,21.686827,83.844810,87.849998,40.886280,25.329765,...,18.725447,50.720333,26.377531,47.878719,30.299231,11.31,40.458683,90.894173,59.250000,42.236214


In [35]:
# Remove tickers with more than 5% NaN values
clean_prices = prices.loc[:, prices.isna().mean() <= 0.05]

# Forward fill short gaps of 1-2 days
clean_prices = clean_prices.ffill(limit=2)
clean_prices = clean_prices.dropna()
clean_prices

Ticker,A,AAPL,ABBV,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2016-01-04,37.569214,23.730946,37.707539,35.380219,21.823122,86.944191,91.970001,44.534782,26.397755,65.637451,...,20.020790,60.409386,26.027582,49.654362,31.806705,12.160000,42.875286,91.126854,66.489998,43.515823
2016-01-05,37.439968,23.136261,37.550438,35.371990,21.902363,87.396690,92.339996,44.207561,26.692379,65.797379,...,20.067774,61.551342,26.290037,50.077450,31.797894,11.510000,42.768421,93.024414,64.820000,44.197056
2016-01-06,37.606152,22.683502,37.556999,35.075294,21.937229,87.225922,91.019997,42.326057,26.044203,64.989632,...,19.624807,58.362644,26.567072,49.660782,31.401188,11.520000,42.465591,93.427193,62.230000,44.206261
2016-01-07,36.008827,21.726154,37.445724,34.234665,21.915041,84.664482,89.110001,41.246212,25.425510,63.006187,...,18.899946,52.872486,26.669157,48.865906,30.590141,11.160000,41.016819,91.279053,59.410000,42.862213
2016-01-08,35.630291,21.841030,36.424656,33.517670,21.686827,83.844810,87.849998,40.886280,25.329765,62.486309,...,18.725447,50.720333,26.377531,47.878719,30.299231,11.310000,40.458683,90.894173,59.250000,42.236214
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,138.073227,273.554016,228.055130,124.180954,96.400002,269.980011,352.980011,276.693634,57.347752,255.984940,...,23.555925,124.747910,72.983070,118.430618,138.261230,66.050003,153.612030,89.990799,245.899994,124.956429
2025-12-26,138.143097,273.144409,228.144394,124.210808,95.870003,271.089996,353.799988,275.975861,57.476780,256.718872,...,23.536098,124.089348,73.308144,118.321342,138.331009,66.269997,152.536987,90.529366,246.270004,125.693283
2025-12-29,137.683914,273.504089,228.997543,123.942169,96.389999,271.339996,353.160004,274.769653,57.784462,257.482544,...,23.605495,122.003899,73.576164,119.731941,137.942322,65.919998,151.571426,90.290001,245.740005,125.444351


## Return Calculations and Universe Construction

In [49]:
daily_returns = np.log(clean_prices/clean_prices.shift(1)).dropna()
# resample to monthly returns
monthly_prices = clean_prices.resample('ME').last()
monthly_returns = monthly_prices.pct_change().dropna()
monthly_returns

Ticker,A,AAPL,ABBV,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2016-02-29,-0.007968,-0.001288,-0.005282,0.023514,0.005774,-0.050029,-0.044654,-0.008057,-0.002009,0.019256,...,0.014448,0.232686,0.034537,0.038888,0.044984,0.190422,0.001382,-0.024683,0.022848,-0.046226
2016-03-31,0.066935,0.127210,0.045962,0.079763,0.046512,0.151008,0.101585,0.117003,0.038616,0.065883,...,0.206172,0.132759,0.066564,0.042919,0.093291,0.463602,0.129433,0.103975,0.116866,0.079640
2016-04-30,0.029823,-0.139921,0.078189,-0.064364,-0.008579,-0.011990,0.004478,-0.048488,0.099973,-0.014157,...,0.036798,-0.054907,-0.042802,0.057543,0.021516,-0.025524,-0.022498,0.085717,-0.093333,0.063114
2016-05-31,0.121457,0.071773,0.031639,0.018766,0.030784,0.053578,0.055721,0.046255,0.079462,-0.006784,...,-0.019302,0.095103,0.033475,0.015477,0.072754,-0.359973,0.031800,0.054764,-0.151055,0.008292
2016-06-30,-0.030794,-0.042660,-0.016208,-0.008074,-0.009083,-0.047743,-0.036996,-0.031795,0.002806,0.052300,...,-0.045433,-0.057601,0.091113,0.053022,-0.000224,-0.050367,0.010111,-0.012169,-0.056675,0.002941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-31,0.094504,0.119639,0.113110,0.051272,0.063560,-0.026694,-0.002768,0.118773,0.166064,-0.017609,...,0.041160,0.165186,-0.014297,0.033263,-0.018444,0.030805,0.019563,0.157665,-0.064686,0.072776
2025-09-30,0.023458,0.096881,0.100475,0.009649,-0.008740,-0.051429,-0.011074,-0.018456,-0.046296,-0.029684,...,-0.041747,0.011992,0.122829,-0.013474,0.041961,-0.092541,0.039245,-0.069358,-0.062853,-0.064450
2025-10-31,0.140319,0.061815,-0.051516,-0.072945,-0.048716,0.014193,-0.035266,-0.047090,0.013224,-0.113118,...,-0.072207,-0.072347,0.006448,0.014279,0.022712,0.050782,-0.090724,0.020914,-0.093922,-0.011812


In [ ]:
fred = Fred(api_key='b7e3b915f2a1c6ca61155ee961eebdf7')
# 3 month treasury bill secondary market rate - standard proxy for risk-free rate
# risk-free rate: the return you'd earn on an investment with 0 risk of loss
tbill = fred.get_series('TB3MS', observation_start='2016-01-01') 
rf_monthly = (tbill / 100 / 12).resample('ME').last() # convert to monthly decimal
rf_monthly

2016-01-31    0.000217
2016-02-29    0.000258
2016-03-31    0.000242
2016-04-30    0.000192
2016-05-31    0.000225
                ...   
2025-10-31    0.003183
2025-11-30    0.003150
2025-12-31    0.002992
2026-01-31    0.002975
2026-02-28    0.003000
Freq: ME, Length: 122, dtype: float64